# Multiscale Transport-Economic Effects

**Purpose.** Assess temporal and scale-dependent heterogeneity in transport-economic associations.

**Inputs.** Country-year transport, macroeconomic, and sectoral data.

**Outputs.** Multiscale estimates, diagnostics, and heatmaps.

Run the code cells from top to bottom after updating the local paths described in the repository README. The notebook has no stored outputs; figures and tables are generated when it is executed.

In [ ]:
# Cell | Dependencies and visual-design settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['xtick.major.width'] = 0.6
plt.rcParams['ytick.major.width'] = 0.6
plt.rcParams['figure.dpi'] = 150
plt.rcParams['axes.unicode_minus'] = False

TRANSPORT_COLS = [
    'railway_length_km',
    'railway_station_area_km2',
    'airport_area_km2',
    'port_area_km2',
    'motorway_trunk_paved_km',
    'total_road_km'
]
MACRO_COLS = ['GDP', 'GDP_per_capita', 'Per_Capita_Consumption']
INDUSTRY_COLS = ['Agriculture_value_added', 'Industry_value_added', 'Services_value_added']
ALL_ECON_COLS = MACRO_COLS + INDUSTRY_COLS
LAG_YEARS = [0, 1, 2, 3]

COUNTRY_TO_REGION = {
    'North Africa':   ['Algeria','Egypt','Libya','Mauritania','Morocco','Tunisia'],
    'West Africa':    ['Benin','Burkina Faso','Ivory Coast','Gambia','Ghana','Guinea',
                       'Guinea-Bissau','Liberia','Mali','Niger','Nigeria','Senegal',
                       'Sierra Leone','Togo'],
    'Central Africa': ['Cameroon','Central African Republic','Chad','Republic of the Congo',
                       'Democratic Republic of the Congo','Equatorial Guinea','Gabon'],
    'East Africa':    ['Burundi','Djibouti','Eritrea','Ethiopia','Kenya',
                       'Rwanda','Somalia','South Sudan','Sudan','United Republic of Tanzania','Uganda'],
    'Southern Africa':['Angola','Botswana','eSwatini','Lesotho','Madagascar','Malawi',
                       'Mozambique','Namibia','South Africa','Zambia','Zimbabwe']
}
REGION_COLORS = {
    'North Africa':   '#E63946',
    'West Africa':    '#F4A261',
    'Central Africa': '#2A9D8F',
    'East Africa':    '#457B9D',
    'Southern Africa':'#6A4C93'
}

region_map = {country: region
              for region, countries in COUNTRY_TO_REGION.items()
              for country in countries}
all_mapped_countries = [c for cs in COUNTRY_TO_REGION.values() for c in cs]
print("Configuration completed ✓")


In [ ]:
# Cell | Country-year transport infrastructure data
transport_file = r'E:\Africa_Infrastructure_Data\transport_statistics_2015_2025.xlsx'
df_transport = pd.read_excel(transport_file)
df_transport.columns = df_transport.columns.str.strip()

road_type_cols = ['road_motorway_km', 'road_trunk_km', 'road_primary_km',
                  'road_secondary_km', 'road_tertiary_km']
df_transport['total_road_km'] = df_transport[road_type_cols].sum(axis=1)
print("Transport data shape:", df_transport.shape)


In [ ]:
# Cell | AfDB macroeconomic data and country-name harmonization
econ_file = r'E:\Africa_Infrastructure_Data\afdb\ObservationData_fhkvkjf_constant.csv'
df_econ_raw = pd.read_csv(econ_file)

country_name_mapping = {
    "Congo, Dem. Rep.": "Democratic Republic of the Congo",
    "Congo, Rep.": "Republic of the Congo",
    "Tanzania": "United Republic of Tanzania",
    "Eswatini": "eSwatini",
    "Cote d'Ivoire": "Ivory Coast"
}
df_econ_raw['country'] = df_econ_raw['country'].replace(country_name_mapping)
df_econ_raw['region'] = df_econ_raw['country'].map(region_map)
df_econ_raw = df_econ_raw[df_econ_raw['country'].isin(all_mapped_countries)].copy()

df_econ_raw['year'] = df_econ_raw['Date'].astype(int)
df_econ_raw['value'] = pd.to_numeric(df_econ_raw['Value'], errors='coerce')
df_econ_raw = df_econ_raw[(df_econ_raw['year'] >= 2015) & (df_econ_raw['year'] <= 2024)]

df_gdp = df_econ_raw[df_econ_raw['indicator'] == 'GDP (constant 2000 US$)'][
    ['country', 'year', 'value']
].rename(columns={'value': 'GDP'})

df_macro = df_gdp.copy()
df_macro['year'] = df_macro['year'].astype('Int64')
print(f"Macroeconomic wide-table shape: {df_macro.shape}")


In [ ]:
# Cell | World Bank sectoral value-added data
import glob
import os

# Load World Bank sectoral value-added series from the local data directory.
wb_dir = r'E:\Africa_Infrastructure_Data\world_bank'
sector_files = {
    'Agriculture_value_added': glob.glob(os.path.join(wb_dir, '**', 'API_NV.AGR*.csv'), recursive=True),
    'Industry_value_added':    glob.glob(os.path.join(wb_dir, '**', 'API_NV.IND*.csv'), recursive=True),
    'Services_value_added':    glob.glob(os.path.join(wb_dir, '**', 'API_NV.SRV*.csv'), recursive=True),
}
wb_to_transport_mapping = {
    'Congo, Dem. Rep.':  'Democratic Republic of the Congo',
    'Congo, Rep.':       'Republic of the Congo',
    'Tanzania':          'United Republic of Tanzania',
    'Eswatini':          'eSwatini',
    "Cote d'Ivoire":     'Ivory Coast',
    'Egypt, Arab Rep.':  'Egypt',
    'Gambia, The':       'Gambia',
}
YEAR_RANGE = list(range(2015, 2025))

def load_sector_long(files, sector_name):
    if not files:
        return pd.DataFrame()
    df = pd.read_csv(files[0], skiprows=4)
    df['Country Name'] = df['Country Name'].replace(wb_to_transport_mapping)
    df = df[df['Country Name'].isin(all_mapped_countries)].copy()
    year_cols = [str(y) for y in YEAR_RANGE if str(y) in df.columns]
    df = df[['Country Name', 'Country Code'] + year_cols]
    valid_countries = df.set_index('Country Name')[year_cols].notna().sum(axis=1)
    valid_countries = valid_countries[valid_countries > 3].index.tolist()
    df = df[df['Country Name'].isin(valid_countries)]
    df_long = df.melt(id_vars=['Country Name', 'Country Code'], value_vars=year_cols,
                      var_name='year', value_name=sector_name)
    df_long['year'] = df_long['year'].astype(int)
    return df_long

dfs = []
for sector, files in sector_files.items():
    df_sector = load_sector_long(files, sector)
    if not df_sector.empty:
        dfs.append(df_sector)

if dfs:
    df_industry = dfs[0]
    for df_s in dfs[1:]:
        df_industry = df_industry.merge(df_s, on=['Country Name', 'Country Code', 'year'], how='outer')
    df_industry = df_industry.rename(columns={'Country Name': 'country'})
    df_industry['year'] = pd.to_numeric(df_industry['year'], errors='coerce').astype('Int64')
    df_industry = df_industry[(df_industry['year'] >= 2015) & (df_industry['year'] <= 2024)]
    print(f"Sectoral data shape: {df_industry.shape}")
else:
    df_industry = pd.DataFrame()
    print("No sectoral data found")


In [ ]:
# Cell | Macro and sectoral multiscale panels
# Work on a copy to preserve the source table.
df_t = df_transport.copy()
df_t['year'] = pd.to_numeric(df_t['year'], errors='coerce').astype('Int64')
df_t = df_t[(df_t['year'] >= 2015) & (df_t['year'] <= 2024)]

# Merge records using the shared country-year keys.
df_panel_macro = df_t.merge(df_macro, on=['country', 'year'], how='left')
df_panel_macro['region'] = df_panel_macro['country'].map(region_map)
df_panel_macro = df_panel_macro.sort_values(['country', 'year']).reset_index(drop=True)

ind_cols = [c for c in df_industry.columns if c not in ['country', 'Country Code', 'year']]
df_panel_industry = df_t.merge(
    df_industry[['country', 'year'] + ind_cols], on=['country', 'year'], how='inner'
)
df_panel_industry['region'] = df_panel_industry['country'].map(region_map)
df_panel_industry = df_panel_industry.sort_values(['country', 'year']).reset_index(drop=True)
print(f"Macro panel: {df_panel_macro.shape}, {df_panel_macro['country'].nunique()} countries")
print(f"Sectoral panel: {df_panel_industry.shape}, {df_panel_industry['country'].nunique()} countries")


In [ ]:
# Cell | Lagged association and confidence-interval functions
# Estimate transport-economic associations over alternative temporal lags.
def compute_lag_correlations(df, transport_col, econ_col, lag_years=LAG_YEARS,
                              min_obs=8, entity_col=None):
    from scipy.stats import norm as sp_norm
    results = {}
    is_industry = econ_col in INDUSTRY_COLS
    if is_industry:
        df = df[(df['year'] >= 2015) & (df['year'] <= 2024)].copy()

    for lag in lag_years:
        if entity_col is None:
            years = sorted(df['year'].unique())
            xs, ys = [], []
            for yr in years:
                if yr + lag not in years:
                    continue
                vx = df.loc[df['year'] == yr,      transport_col].values
                vy = df.loc[df['year'] == yr + lag, econ_col].values
                if len(vx) > 0 and len(vy) > 0 and pd.notna(vx[0]) and pd.notna(vy[0]):
                    xs.append(vx[0]); ys.append(vy[0])
            if len(xs) < min_obs or np.std(xs) == 0 or np.std(ys) == 0:
                results[lag] = {'r': np.nan, 'p': np.nan, 'n': len(xs)}
            else:
                _, _, rv, pv, _ = stats.linregress(xs, ys)
                results[lag] = {'r': rv, 'p': pv, 'n': len(xs)}
        else:
            r_list, p_list, n_total = [], [], 0
            for ent in df[entity_col].unique():
                df_e = df[df[entity_col] == ent]
                years = sorted(df_e['year'].unique())
                xs, ys = [], []
                for yr in years:
                    if yr + lag not in years:
                        continue
                    vx = df_e.loc[df_e['year'] == yr,      transport_col].values
                    vy = df_e.loc[df_e['year'] == yr + lag, econ_col].values
                    if len(vx) > 0 and len(vy) > 0 and pd.notna(vx[0]) and pd.notna(vy[0]):
                        xs.append(vx[0]); ys.append(vy[0])
                if len(xs) < 4 or np.std(xs) == 0 or np.std(ys) == 0:
                    continue
                _, _, rv, pv, _ = stats.linregress(xs, ys)
                r_list.append(rv); p_list.append(pv); n_total += len(xs)
            if len(r_list) < 1:
                results[lag] = {'r': np.nan, 'p': np.nan, 'n': 0}
            else:
                z_fisher = [0.5 * np.log((1 + r) / (1 - r)) for r in r_list
                            if not np.isnan(r) and abs(r) < 1]
                if len(z_fisher) == 0:
                    r_mean = float(np.mean(r_list))
                else:
                    z_bar = float(np.mean(z_fisher))
                    e2z = np.exp(2 * z_bar)
                    r_mean = float((e2z - 1) / (e2z + 1))
                z_scores = [sp_norm.ppf(1 - min(pv, 0.9999)) for pv in p_list]
                z = float(np.sum(z_scores) / np.sqrt(len(z_scores)))
                p_combined = float(1 - sp_norm.cdf(z))
                results[lag] = {'r': r_mean, 'p': p_combined, 'n': n_total}
    return results


def build_lag_matrix(df, transport_cols, econ_cols, lag_years=LAG_YEARS,
                     min_obs=8, entity_col=None):
    col_index = pd.MultiIndex.from_product([transport_cols, lag_years],
                                            names=['transport', 'lag'])
    r_mat = pd.DataFrame(index=econ_cols, columns=col_index, dtype=float)
    p_mat = pd.DataFrame(index=econ_cols, columns=col_index, dtype=float)
    for t_col in transport_cols:
        if t_col not in df.columns:
            continue
        for e_col in econ_cols:
            if e_col not in df.columns:
                continue
            res = compute_lag_correlations(df, t_col, e_col, lag_years,
                                           min_obs=min_obs, entity_col=entity_col)
            for lag, vals in res.items():
                r_mat.loc[e_col, (t_col, lag)] = vals['r']
                p_mat.loc[e_col, (t_col, lag)] = vals['p']
    return r_mat.astype(float), p_mat.astype(float)
print("Analysis functions defined ✓")


In [ ]:
# Cell | Correlation-heatmap inputs and labels

from mpl_toolkits.axes_grid1.inset_locator import inset_axes as _inset_axes

use_transport = [c for c in TRANSPORT_COLS if c in df_panel_macro.columns]

_ECON_SHORT = {
    'GDP':                      'GDP',
    'Agriculture_value_added':  'Agriculture',
    'Industry_value_added':     'Industry',
    'Services_value_added':     'Services',
}
_TRANSPORT_ORDER = [
    'total_road_km',
    'motorway_trunk_paved_km',
    'railway_length_km',
    'railway_station_area_km2',
    'port_area_km2',
    'airport_area_km2',
]
_TRANSPORT_SHORT = {
    'total_road_km':            'Road',
    'motorway_trunk_paved_km':  'Paved Hwy',
    'railway_length_km':        'Railway',
    'railway_station_area_km2': 'Rly Station',
    'port_area_km2':            'Port',
    'airport_area_km2':         'Aviation',
}
_PANEL_LABELS = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)']

_t_cols = [c for c in _TRANSPORT_ORDER if c in use_transport]
_use_macro_cols    = ['GDP']
_use_industry_cols = [c for c in INDUSTRY_COLS if c in df_panel_industry.columns]
_e_cols = _use_macro_cols + _use_industry_cols
_n_t, _n_e = len(_t_cols), len(_e_cols)

_cmap = plt.cm.RdBu_r
_norm = Normalize(vmin=-1, vmax=1)
_lag_plot = [0, 1, 2, 3]

_FS_ECON      = 20
_FS_TRANS     = 20
_FS_SUBTITLE  = 24
_FS_CBAR      = 26
_FS_CBAR_TICK = 20
_FS_STAR      = 20
_STAR_OFFSET_PT = 4

_macro_cols    = _use_macro_cols
_industry_cols = _use_industry_cols
_SUM_COLS_GR   = ['GDP'] + INDUSTRY_COLS

def _agg_africa(df_src, econ_cols):
    t_agg = {c: 'sum' for c in _t_cols if c in df_src.columns}
    df_a = df_src.groupby('year').agg(t_agg).reset_index()
    for ec in econ_cols:
        if ec in df_src.columns:
            df_a[ec] = df_src.groupby('year')[ec].sum().values if ec in _SUM_COLS_GR \
                       else df_src.groupby('year')[ec].mean().values
    return df_a

def _agg_regional(df_src, econ_cols):
    t_cols_avail = [c for c in _t_cols if c in df_src.columns]
    e_cols_avail = [c for c in econ_cols if c in df_src.columns]
    agg_dict = {c: 'mean' for c in t_cols_avail}
    agg_dict.update({c: ('mean' if c in _SUM_COLS_GR else 'mean') for c in e_cols_avail})
    return df_src.groupby(['region', 'year']).agg(agg_dict).reset_index()

def _compute_growth_rates(df_src, value_cols, entity_col):
    _PORT_INVALID_COUNTRIES = ['Guinea', 'Ethiopia', 'Zambia']
    
    records = []
    for ent in df_src[entity_col].unique():
        df_e = df_src[df_src[entity_col] == ent].sort_values('year')
        base_row = df_e[df_e['year'] == 2015]
        if base_row.empty:
            continue
        base = base_row.iloc[0]
        for _, row in df_e.iterrows():
            if row['year'] == 2015:
                continue
            rec = {'year': row['year'], entity_col: ent}
            for c in value_cols:
                if c == 'port_area_km2' and ent in _PORT_INVALID_COUNTRIES:
                    rec[f'{c}_gr'] = np.nan
                    continue
                
                bv = base[c]
                if pd.notna(bv) and bv != 0:
                    rec[f'{c}_gr'] = (row[c] - bv) / abs(bv) * 100
                elif pd.notna(bv) and bv == 0:
                    rec[f'{c}_gr'] = np.nan
                else:
                    rec[f'{c}_gr'] = np.nan
            records.append(rec)
    return pd.DataFrame(records)

def _agg_then_gr(df_src, t_cols, econ_cols):
    agg_dict = {}
    for c in t_cols + econ_cols:
        if c in df_src.columns:
            agg_dict[c] = 'sum' if (c in _SUM_COLS_GR or c in t_cols) else 'mean'
    df_agg = df_src.groupby('year').agg(agg_dict).reset_index()
    df_agg['_entity'] = 'Africa'
    return _compute_growth_rates(df_agg, [c for c in t_cols + econ_cols if c in df_agg.columns],
                                  entity_col='_entity')

def _regional_then_gr(df_src, t_cols, econ_cols):
    t_cols_avail = [c for c in t_cols if c in df_src.columns]
    e_cols_avail = [c for c in econ_cols if c in df_src.columns]
    agg_dict = {c: 'mean' for c in t_cols_avail}
    agg_dict.update({c: ('mean' if c in _SUM_COLS_GR else 'mean') for c in e_cols_avail})
    df_reg = df_src.groupby(['region', 'year']).agg(agg_dict).reset_index()
    return _compute_growth_rates(df_reg, t_cols_avail + e_cols_avail, entity_col='region')

def _build_gr_lag_matrix(df_gr, t_gr_cols, e_gr_cols, lag_years=LAG_YEARS,
                          min_obs=3, entity_col=None):
    t_orig_cols = [c.replace('_gr', '') for c in t_gr_cols]
    e_orig_cols = [c.replace('_gr', '') for c in e_gr_cols]
    col_index = pd.MultiIndex.from_product([t_orig_cols, lag_years],
                                            names=['transport', 'lag'])
    r_mat = pd.DataFrame(index=e_orig_cols, columns=col_index, dtype=float)
    p_mat = pd.DataFrame(index=e_orig_cols, columns=col_index, dtype=float)
    for t_gr, t_orig in zip(t_gr_cols, t_orig_cols):
        if t_gr not in df_gr.columns:
            continue
        for e_gr, e_orig in zip(e_gr_cols, e_orig_cols):
            if e_gr not in df_gr.columns:
                continue
            df_tmp = df_gr.rename(columns={t_gr: t_orig, e_gr: e_orig})
            res = compute_lag_correlations(df_tmp, t_orig, e_orig, lag_years,
                                           min_obs=min_obs, entity_col=entity_col)
            for lag, vals in res.items():
                r_mat.loc[e_orig, (t_orig, lag)] = vals['r']
                p_mat.loc[e_orig, (t_orig, lag)] = vals['p']
    return r_mat.astype(float), p_mat.astype(float)

_trans_gr_cols = [f'{c}_gr' for c in _t_cols]
_macro_gr_cols = [f'{c}_gr' for c in _macro_cols]
_ind_gr_cols   = [f'{c}_gr' for c in _industry_cols]

_df_africa_gr_m = _agg_then_gr(df_panel_macro,    _t_cols, _macro_cols)
_df_africa_gr_i = _agg_then_gr(df_panel_industry, _t_cols, _industry_cols)
_df_reg_gr_m    = _regional_then_gr(df_panel_macro,    _t_cols, _macro_cols)
_df_reg_gr_i    = _regional_then_gr(df_panel_industry, _t_cols, _industry_cols)
_df_nat_gr_m    = _compute_growth_rates(
    df_panel_macro[['country', 'year'] + _t_cols + _macro_cols].copy(),
    _t_cols + _macro_cols, entity_col='country')
_df_nat_gr_i    = _compute_growth_rates(
    df_panel_industry[['country', 'year'] + _t_cols + _industry_cols].copy(),
    _t_cols + _industry_cols, entity_col='country')


print(f"Growth-rate data year range: {sorted(_df_africa_gr_m['year'].unique())}")
_r_gr_am, _p_gr_am = _build_gr_lag_matrix(_df_africa_gr_m, _trans_gr_cols, _macro_gr_cols, min_obs=3, entity_col=None)
_r_gr_ai, _p_gr_ai = _build_gr_lag_matrix(_df_africa_gr_i, _trans_gr_cols, _ind_gr_cols,   min_obs=3, entity_col=None)
_r_gr_africa = pd.concat([_r_gr_am, _r_gr_ai]); _p_gr_africa = pd.concat([_p_gr_am, _p_gr_ai])

_r_gr_rm, _p_gr_rm = _build_gr_lag_matrix(_df_reg_gr_m, _trans_gr_cols, _macro_gr_cols, min_obs=3, entity_col='region')
_r_gr_ri, _p_gr_ri = _build_gr_lag_matrix(_df_reg_gr_i, _trans_gr_cols, _ind_gr_cols,   min_obs=3, entity_col='region')
_r_gr_regional = pd.concat([_r_gr_rm, _r_gr_ri]); _p_gr_regional = pd.concat([_p_gr_rm, _p_gr_ri])

_r_gr_nm, _p_gr_nm = _build_gr_lag_matrix(_df_nat_gr_m, _trans_gr_cols, _macro_gr_cols, min_obs=4, entity_col='country')
_r_gr_ni, _p_gr_ni = _build_gr_lag_matrix(_df_nat_gr_i, _trans_gr_cols, _ind_gr_cols,   min_obs=4, entity_col='country')
_r_gr_national = pd.concat([_r_gr_nm, _r_gr_ni]); _p_gr_national = pd.concat([_p_gr_nm, _p_gr_ni])

_regions = [r for r in COUNTRY_TO_REGION.keys() if r in df_panel_macro['region'].unique()]
_r_gr_regs, _p_gr_regs = {}, {}
for _reg in _regions:
    _rm_g = _df_reg_gr_m[_df_reg_gr_m['region'] == _reg].copy()
    _ri_g = _df_reg_gr_i[_df_reg_gr_i['region'] == _reg].copy()
    _rrm_g, _rpm_g = _build_gr_lag_matrix(_rm_g, _trans_gr_cols, _macro_gr_cols, min_obs=3, entity_col=None)
    _rri_g, _rpi_g = _build_gr_lag_matrix(_ri_g, _trans_gr_cols, _ind_gr_cols,   min_obs=3, entity_col=None)
    _r_gr_regs[_reg] = pd.concat([_rrm_g, _rri_g]); _p_gr_regs[_reg] = pd.concat([_rpm_g, _rpi_g])

_gr_panels = (
    [('Continental', _r_gr_africa, _p_gr_africa)]
    + [(_reg, _r_gr_regs[_reg], _p_gr_regs[_reg]) for _reg in _regions]
    + [('National', _r_gr_national, _p_gr_national)]
)
_n_panels = len(_gr_panels)
_COLS_PER_ROW = 2
_n_rows = int(np.ceil(_n_panels / _COLS_PER_ROW))

_cell_w      = 1.5
_cell_h      = 0.5
_label_col_w = 1.6
_no_label_w  = 0.10
_col_label_h = 0.40
_subtitle_h  = 0.45

_panel_content_h = _subtitle_h + _col_label_h + _cell_h * _n_e
_panel_w_left    = _label_col_w + _cell_w * _n_t
_panel_w_right   = _no_label_w  + _cell_w * _n_t

_gap_x_in    = 0.00
_gap_y_in    = -0.2
_margin_l_in = 0.15
_margin_r_in = 0.15
_margin_b_in = 0.10
_margin_t_in = 0.10

_fig_w = _panel_w_left + _panel_w_right + _gap_x_in + _margin_l_in + _margin_r_in
_fig_h = (_panel_content_h * _n_rows + _gap_y_in * (_n_rows - 1) + _margin_b_in + _margin_t_in)

def _ix(x): return x / _fig_w
def _iy(y): return y / _fig_h

def _draw_panel(fig, left_in, bottom_in, r_mat, p_mat, label, title,
                show_col_labels=False, show_row_labels=False):
    lw    = _label_col_w if show_row_labels else _no_label_w
    col_h = _col_label_h if show_col_labels else 0.0

    sub_y_in = bottom_in + _cell_h * _n_e + col_h + _subtitle_h * 0.4
    sub_x_in = left_in if show_row_labels else left_in + lw
    fig.text(_ix(sub_x_in), _iy(sub_y_in), f'{label} {title}',
             ha='left', va='center', fontsize=_FS_SUBTITLE, fontweight='bold',
             transform=fig.transFigure)

    if show_col_labels:
        col_y_in = bottom_in + _cell_h * _n_e + col_h * 0.3
        for ci, t_col in enumerate(_t_cols):
            x_in = left_in + lw + (ci + 0.5) * _cell_w
            fig.text(_ix(x_in), _iy(col_y_in), _TRANSPORT_SHORT.get(t_col, t_col),
                     ha='center', va='center', fontsize=_FS_TRANS,
                     transform=fig.transFigure)

    for ri, e_col in enumerate(_e_cols):
        if show_row_labels:
            y_in = bottom_in + (_n_e - ri - 0.5) * _cell_h
            fig.text(_ix(left_in + lw - 0.08), _iy(y_in),
                     _ECON_SHORT.get(e_col, e_col),
                     ha='right', va='center', fontsize=_FS_ECON,
                     transform=fig.transFigure)

        for ci, t_col in enumerate(_t_cols):
            if e_col not in r_mat.index:
                continue
            r0 = r_mat.loc[e_col, (t_col, 0)] if (t_col, 0) in r_mat.columns else np.nan
            ax = fig.add_axes([
                _ix(left_in + lw + ci * _cell_w),
                _iy(bottom_in + (_n_e - ri - 1) * _cell_h),
                _ix(_cell_w), _iy(_cell_h)
            ])
            bg_color = _cmap(_norm(r0)) if not np.isnan(r0) else (0.85, 0.85, 0.85, 1)
            ax.set_facecolor(bg_color)
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_linewidth(0.5); sp.set_color('white')

            lum = 0.299*bg_color[0] + 0.587*bg_color[1] + 0.114*bg_color[2]
            line_color = 'white' if lum < 0.45 else '#222222'

            r_vals = [r_mat.loc[e_col, (t_col, lg)] if (t_col, lg) in r_mat.columns else np.nan
                      for lg in _lag_plot]
            p_vals = [p_mat.loc[e_col, (t_col, lg)] if (t_col, lg) in p_mat.columns else np.nan
                      for lg in _lag_plot]

            if not all(np.isnan(r_vals)):
                iax = _inset_axes(ax, width='100%', height='100%', loc='center',
                                  bbox_to_anchor=(0.0, 0.0, 1.0, 1.0),
                                  bbox_transform=ax.transAxes, borderpad=0)
                iax.set_facecolor('none'); iax.patch.set_alpha(0)
                valid = [(x, y) for x, y in zip(_lag_plot, r_vals) if not np.isnan(y)]
                if len(valid) >= 2:
                    xs, ys = zip(*valid)
                    iax.plot(xs, ys, '-o', color=line_color, linewidth=1.6,
                             markersize=5, markeredgewidth=0)
                iax.set_xlim(-0.15, 3.15); iax.set_ylim(-1.2, 1.2)
                iax.axhline(0, color=line_color, linewidth=1.0, alpha=0.55, linestyle='--')
                iax.set_xticks([]); iax.set_yticks([])
                for sp in iax.spines.values(): sp.set_visible(False)
                for lg, rv, pv in zip(_lag_plot, r_vals, p_vals):
                    if np.isnan(rv) or np.isnan(pv) or pv >= 0.05: continue
                    va = 'top' if rv >= 0 else 'bottom'
                    iax.annotate('*', xy=(lg, rv), xycoords='data',
                                 xytext=(0, -_STAR_OFFSET_PT), textcoords='offset points',
                                 ha='center', va=va, fontsize=_FS_STAR,
                                 color=line_color, annotation_clip=True)

fig10 = plt.figure(figsize=(_fig_w, _fig_h), facecolor='white')

for idx, (title, r_mat, p_mat) in enumerate(_gr_panels):
    row = idx // _COLS_PER_ROW
    col = idx %  _COLS_PER_ROW
    left_in   = _margin_l_in if col == 0 else _margin_l_in + _panel_w_left + _gap_x_in
    bottom_in = _margin_b_in + (_n_rows - row - 1) * (_panel_content_h + _gap_y_in)
    _draw_panel(fig10, left_in, bottom_in, r_mat, p_mat,
                label=_PANEL_LABELS[idx], title=title,
                show_col_labels=(row == 0),
                show_row_labels=(col == 0))

# ── colorbar ──────────────────────────────────────────────────
sm10 = plt.cm.ScalarMappable(cmap=_cmap, norm=_norm)
sm10.set_array([])
_total_h = _panel_content_h * _n_rows + _gap_y_in * (_n_rows - 1)
_cbar_h  = _total_h / 3
_cbar_b  = _margin_b_in
_cbar_l  = _fig_w - _margin_r_in + 0.35
_cbar_w  = 0.25
cbar10_ax = fig10.add_axes([_ix(_cbar_l), _iy(_cbar_b), _ix(_cbar_w), _iy(_cbar_h)])
cbar10 = fig10.colorbar(sm10, cax=cbar10_ax)
cbar10.set_label('Sync r', fontsize=_FS_CBAR, fontweight='bold', labelpad=8)
cbar10.ax.tick_params(labelsize=_FS_CBAR_TICK)
cbar10.set_ticks([-1, -0.5, 0, 0.5, 1])

plt.show()
print("Three-scale growth-rate heatmap completed ✓")


In [ ]:
# Cell | GDP growth patterns across spatial scales

from matplotlib.lines import Line2D

gdp_base_country = df_macro[df_macro['year'] == 2015][['country', 'GDP']].set_index('country')
gdp_2024 = df_macro[df_macro['year'] == 2024][['country', 'GDP']].set_index('country')

gdp_growth_rate = ((gdp_2024['GDP'] - gdp_base_country['GDP']) / gdp_base_country['GDP'] * 100) / 9
gdp_growth_rate = gdp_growth_rate.reset_index()
gdp_growth_rate.columns = ['country', 'avg_growth_rate']
gdp_growth_rate['region'] = gdp_growth_rate['country'].map(region_map)

region_order_for_top = ['North Africa', 'West Africa', 'Central Africa', 'East Africa', 'Southern Africa']
top_countries = []
top_countries_info = {}

print("Countries with the fastest GDP growth in the five regions:")
for region in region_order_for_top:
    region_data = gdp_growth_rate[gdp_growth_rate['region'] == region]
    if not region_data.empty:
        top_country = region_data.loc[region_data['avg_growth_rate'].idxmax()]
        top_countries.append(top_country['country'])
        top_countries_info[region] = {
            'country': top_country['country'],
            'growth_rate': top_country['avg_growth_rate']
        }

        print(f"  {region}: {top_country['country']} (Average annual growth rate: {top_country['avg_growth_rate']:.2f}%)")
_r_gr_top_countries = {}
_p_gr_top_countries = {}

for region in region_order_for_top:
    if region in top_countries_info:
        country = top_countries_info[region]['country']
        _nm_g_country = _df_nat_gr_m[_df_nat_gr_m['country'] == country].copy()
        _ni_g_country = _df_nat_gr_i[_df_nat_gr_i['country'] == country].copy()
        _rrm_g_c, _rpm_g_c = _build_gr_lag_matrix(_nm_g_country, _trans_gr_cols, _macro_gr_cols, min_obs=3, entity_col=None)
        _rri_g_c, _rpi_g_c = _build_gr_lag_matrix(_ni_g_country, _trans_gr_cols, _ind_gr_cols,   min_obs=3, entity_col=None)
        _r_gr_top_countries[country] = pd.concat([_rrm_g_c, _rri_g_c])
        _p_gr_top_countries[country] = pd.concat([_rpm_g_c, _rpi_g_c])

_top_NA = top_countries_info['North Africa']['country']
_top_WA = top_countries_info['West Africa']['country']
_top_CA = top_countries_info['Central Africa']['country']
_top_EA = top_countries_info['East Africa']['country']
_top_SA = top_countries_info['Southern Africa']['country']

_gr_panels_new = [
    ('Continental',    _r_gr_africa,                    _p_gr_africa,                    'A'),
    ('North Africa',   _r_gr_regs['North Africa'],      _p_gr_regs['North Africa'],      'C'),
    ('West Africa',    _r_gr_regs['West Africa'],        _p_gr_regs['West Africa'],        'D'),
    ('National',       _r_gr_national,                  _p_gr_national,                  'B'),
    (_top_NA,          _r_gr_top_countries[_top_NA],    _p_gr_top_countries[_top_NA],    ''),
    (_top_WA,          _r_gr_top_countries[_top_WA],    _p_gr_top_countries[_top_WA],    ''),
    ('Central Africa', _r_gr_regs['Central Africa'],    _p_gr_regs['Central Africa'],    'E'),
    ('East Africa',    _r_gr_regs['East Africa'],        _p_gr_regs['East Africa'],        'F'),
    ('Southern Africa',_r_gr_regs['Southern Africa'],   _p_gr_regs['Southern Africa'],   'G'),
    (_top_CA,          _r_gr_top_countries[_top_CA],    _p_gr_top_countries[_top_CA],    ''),
    (_top_EA,          _r_gr_top_countries[_top_EA],    _p_gr_top_countries[_top_EA],    ''),
    (_top_SA,          _r_gr_top_countries[_top_SA],    _p_gr_top_countries[_top_SA],    ''),
]
_COLS_PER_ROW_N = 3
_N_ROWS_N       = 4

_ECON_SHORT_N = {
    'GDP':                      'GDP',
    'Agriculture_value_added':  'AVA',
    'Industry_value_added':     'IVA',
    'Services_value_added':     'SVA',
}
_TRANSPORT_SHORT_N = {
    'total_road_km':            'Road',
    'motorway_trunk_paved_km':  'P.H.',
    'railway_length_km':        'Rail.',
    'railway_station_area_km2': 'Rail.S.',
    'port_area_km2':            'Port',
    'airport_area_km2':         'Aviation',
}

_cell_w_N       = 1.5
_cell_h_N       = 0.8
_label_col_w_N  = 1.2
_no_label_w_N   = 0.10
_col_label_h_N  = 0.40
_subtitle_h_N   = 0.45

_panel_content_h_N = _subtitle_h_N + _col_label_h_N + _cell_h_N * _n_e
_panel_w_left_N    = _label_col_w_N + _cell_w_N * _n_t
_panel_w_mid_N     = _no_label_w_N  + _cell_w_N * _n_t

_gap_x_N         = 0.50
_gap_y_normal_N  = -0.20
_gap_y_special_N =  0.2

_margin_l_N = 0.05
_margin_r_N = 0.15
_margin_b_N = 0.10
_margin_t_N = 0.10

_row_bot = {}
_row_bot[3] = _margin_b_N
_row_bot[2] = _row_bot[3] + _panel_content_h_N + _gap_y_normal_N
_row_bot[1] = _row_bot[2] + _panel_content_h_N + _gap_y_special_N
_row_bot[0] = _row_bot[1] + _panel_content_h_N + _gap_y_normal_N

_fig_w_N = _panel_w_left_N + 2 * _panel_w_mid_N + 2 * _gap_x_N + _margin_l_N + _margin_r_N
_fig_h_N = _row_bot[0] + _panel_content_h_N + _margin_t_N

def _iX(x): return x / _fig_w_N
def _iY(y): return y / _fig_h_N

_col_left = {
    0: _margin_l_N,
    1: _margin_l_N + _panel_w_left_N + _gap_x_N,
    2: _margin_l_N + _panel_w_left_N + _gap_x_N + _panel_w_mid_N + _gap_x_N,
}

_FS_ECON_N      = 32
_FS_TRANS_N     = 32
_FS_SUBTITLE_N  = 36
_FS_LABEL_N     = 42
_FS_CBAR_N      = 32
_FS_CBAR_TICK_N = 32
_FS_STAR_N      = 34
_STAR_OFF_N     = 5
_MSIZE_N        = 10
_LW_N           = 2.5

def _draw_panel_N(fig, left_in, bottom_in, r_mat, p_mat, letter, title,
                  show_col_labels=False, show_row_labels=False):
    lw    = _label_col_w_N if show_row_labels else _no_label_w_N
    col_h = _col_label_h_N if show_col_labels else 0.0

    sub_y_in = bottom_in + _cell_h_N * _n_e + col_h + _subtitle_h_N * 0.4
    sub_x_in = left_in if show_row_labels else left_in + lw

    if letter:
        fig.text(_iX(sub_x_in), _iY(sub_y_in), letter,
                 ha='left', va='center', fontsize=_FS_LABEL_N, fontweight='bold',
                 transform=fig.transFigure)
        letter_w = len(letter) * _FS_LABEL_N * 0.012
        fig.text(_iX(sub_x_in + letter_w), _iY(sub_y_in), f' {title}',
                 ha='left', va='center', fontsize=_FS_SUBTITLE_N, fontweight='bold',
                 transform=fig.transFigure)
    else:
        fig.text(_iX(sub_x_in), _iY(sub_y_in), title,
                 ha='left', va='center', fontsize=_FS_SUBTITLE_N, fontweight='bold',
                 transform=fig.transFigure)

    if show_col_labels:
        col_y_in = bottom_in + _cell_h_N * _n_e + col_h * 0.4
        for ci, t_col in enumerate(_t_cols):
            x_in = left_in + lw + (ci + 0.5) * _cell_w_N
            fig.text(_iX(x_in), _iY(col_y_in), _TRANSPORT_SHORT_N.get(t_col, t_col),
                     ha='center', va='center', fontsize=_FS_TRANS_N,
                     transform=fig.transFigure)

    for ri, e_col in enumerate(_e_cols):
        if show_row_labels:
            y_in = bottom_in + (_n_e - ri - 0.5) * _cell_h_N
            fig.text(_iX(left_in + lw - 0.05), _iY(y_in),
                     _ECON_SHORT_N.get(e_col, e_col),
                     ha='right', va='center', fontsize=_FS_ECON_N,
                     transform=fig.transFigure)

        for ci, t_col in enumerate(_t_cols):
            if e_col not in r_mat.index:
                continue
            r0 = r_mat.loc[e_col, (t_col, 0)] if (t_col, 0) in r_mat.columns else np.nan
            ax = fig.add_axes([
                _iX(left_in + lw + ci * _cell_w_N),
                _iY(bottom_in + (_n_e - ri - 1) * _cell_h_N),
                _iX(_cell_w_N), _iY(_cell_h_N)
            ])
            bg_color = _cmap(_norm(r0)) if not np.isnan(r0) else (0.85, 0.85, 0.85, 1)
            ax.set_facecolor(bg_color)
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_linewidth(0.5); sp.set_color('white')

            lum = 0.299*bg_color[0] + 0.587*bg_color[1] + 0.114*bg_color[2]
            line_color = 'white' if lum < 0.45 else '#222222'

            r_vals = [r_mat.loc[e_col, (t_col, lg)] if (t_col, lg) in r_mat.columns else np.nan
                      for lg in _lag_plot]
            p_vals = [p_mat.loc[e_col, (t_col, lg)] if (t_col, lg) in p_mat.columns else np.nan
                      for lg in _lag_plot]

            if not all(np.isnan(r_vals)):
                iax = _inset_axes(ax, width='100%', height='100%', loc='center',
                                  bbox_to_anchor=(0.0, 0.0, 1.0, 1.0),
                                  bbox_transform=ax.transAxes, borderpad=0)
                iax.set_facecolor('none'); iax.patch.set_alpha(0)
                valid = [(x, y) for x, y in zip(_lag_plot, r_vals) if not np.isnan(y)]
                if len(valid) >= 2:
                    xs, ys = zip(*valid)
                    iax.plot(xs, ys, '-o', color=line_color, linewidth=_LW_N,
                             markersize=_MSIZE_N, markeredgewidth=0)
                iax.set_xlim(-0.5, 3.5); iax.set_ylim(-1.2, 1.2)
                iax.axhline(0, color=line_color, linewidth=1.0, alpha=0.55, linestyle='--')
                iax.set_xticks([]); iax.set_yticks([])
                for sp in iax.spines.values(): sp.set_visible(False)
                for lg, rv, pv in zip(_lag_plot, r_vals, p_vals):
                    if np.isnan(rv) or np.isnan(pv) or pv >= 0.05: continue
                    va = 'top' if rv >= 0 else 'bottom'
                    iax.annotate('*', xy=(lg, rv), xycoords='data',
                                 xytext=(0, -_STAR_OFF_N), textcoords='offset points',
                                 ha='center', va=va, fontsize=_FS_STAR_N,
                                 color=line_color, annotation_clip=True)

fig_N = plt.figure(figsize=(_fig_w_N, _fig_h_N), facecolor='white')

for idx, (title, r_mat, p_mat, letter) in enumerate(_gr_panels_new):
    row_idx = idx // _COLS_PER_ROW_N
    col_idx = idx %  _COLS_PER_ROW_N
    left_in   = _col_left[col_idx]
    bottom_in = _row_bot[row_idx]

    _draw_panel_N(fig_N, left_in, bottom_in, r_mat, p_mat, letter, title,
                  show_col_labels=(row_idx == 0),
                  show_row_labels=(col_idx == 0))

_y_bot_fig = _iY(_margin_b_N)
_y_top_fig = _iY(_row_bot[0] + _panel_content_h_N)

for col_gap in [0, 1]:
    x_in = _col_left[col_gap + 1] - _gap_x_N / 2
    x_fig = _iX(x_in)
    vline = Line2D([x_fig, x_fig], [_y_bot_fig, _y_top_fig],
                   transform=fig_N.transFigure, clip_on=False,
                   color='gray', linewidth=3, linestyle='--', alpha=0.6)
    fig_N.add_artist(vline)

_y_hline_in = _row_bot[2] + _panel_content_h_N - _gap_y_special_N *0.5
_y_hline_fig = _iY(_y_hline_in)
_x_left_fig  = _iX(_margin_l_N)
_x_right_fig = _iX(_fig_w_N - _margin_r_N)
hline = Line2D([_x_left_fig, _x_right_fig], [_y_hline_fig, _y_hline_fig],
               transform=fig_N.transFigure, clip_on=False,
               color='gray', linewidth=3, linestyle='--', alpha=0.6)
fig_N.add_artist(hline)

# ── colorbar ──────────────────────────────────────────────────
sm_N = plt.cm.ScalarMappable(cmap=_cmap, norm=_norm)
sm_N.set_array([])
_total_h_N = _row_bot[0] + _panel_content_h_N - _margin_b_N
_cbar_h_N  = _total_h_N / 3
_cbar_b_N  = _margin_b_N
_cbar_l_N  = _fig_w_N - _margin_r_N + 0.35
_cbar_w_N  = 0.25
cbar_N_ax = fig_N.add_axes([_iX(_cbar_l_N), _iY(_cbar_b_N), _iX(_cbar_w_N), _iY(_cbar_h_N)])
cbar_N = fig_N.colorbar(sm_N, cax=cbar_N_ax)
cbar_N.set_label('Sync r', fontsize=_FS_CBAR_N, labelpad=8)
cbar_N.ax.tick_params(labelsize=_FS_CBAR_TICK_N)
cbar_N.set_ticks([-1, -0.5, 0, 0.5, 1])

plt.show()
print("Three-scale heatmap completed (new 3-column by 4-row layout with separators) ✓")


In [ ]:
# Cell | Robust OLS helper for scale-specific estimates

# Fit OLS models with heteroskedasticity-consistent HC1 standard errors.
def _ols_hc1(X, y):
    """Fit OLS with heteroskedasticity-consistent HC1 standard errors."""
    try:
        from scipy import stats as sp_stats
        
        n, k = X.shape
        
        XtX_inv = np.linalg.inv(X.T @ X)
        coef = XtX_inv @ X.T @ y
        
        # Calculate residual-based diagnostic quantities.
        resid = y - X @ coef
        
        meat = np.zeros((k, k))
        for i in range(n):
            meat += resid[i]**2 * np.outer(X[i], X[i])
        
        hc1_factor = n / (n - k)
        vcov = hc1_factor * XtX_inv @ meat @ XtX_inv
        se = np.sqrt(np.diag(vcov))
        
        t_stat = coef / se
        pval = 2 * (1 - sp_stats.t.cdf(np.abs(t_stat), df=n-k))
        
        return coef, se, pval
        
    except Exception as e:
        print(f"OLS regression failed: {str(e)}")
        return None, None, None
print("OLS regression function defined ✓")


In [ ]:
# Cell | Model A: lagged scale-specific associations

# Evaluate lagged effects up to three years at each spatial scale.
MAX_LAG_A = 3
_REGIONS_A = ['North Africa', 'West Africa', 'Central Africa',
              'East Africa', 'Southern Africa']
_REF_REGION_A = 'North Africa'

_LAG_LABELS = ['L0', 'L1', 'L2', 'L3']
_REGION_ABBR = {
    'North Africa':   'N.Afr',
    'West Africa':    'W.Afr',
    'Central Africa': 'C.Afr',
    'East Africa':    'E.Afr',
    'Southern Africa':'S.Afr'
}
_ECON_LABEL_A = {
    'GDP':                     'GDP',
    'Agriculture_value_added': 'Agriculture',
    'Industry_value_added':    'Industry',
    'Services_value_added':    'Services'
}
_TRANS_LABEL_A = {
    'total_road_km':            'Road',
    'motorway_trunk_paved_km':  'Paved Hwy',
    'railway_length_km':        'Railway',
    'railway_station_area_km2': 'Rly Station',
    'port_area_km2':            'Port',
    'airport_area_km2':         'Aviation'
}

def _build_lag_panel_A_gr(df_gr, t_gr_cols, e_gr_cols, max_lag=MAX_LAG_A):
    """Helper function."""
    df = df_gr.copy().sort_values(['country', 'year'])
    df['year'] = df['year'].astype(int)

    for tc_gr in t_gr_cols + e_gr_cols:
        if tc_gr not in df.columns:
            continue
        for lag in range(max_lag + 1):
            df[f'{tc_gr}_L{lag}'] = df.groupby('country')[tc_gr].shift(lag)
    return df

def _run_region_interact_model_gr(df_panel_gr, t_gr_col, e_gr_col, max_lag=MAX_LAG_A):
    """Helper function."""
    lag_cols = [f'{t_gr_col}_L{lag}' for lag in range(max_lag + 1)]
    needed = [e_gr_col, 'country', 'year', 'region'] + lag_cols
    df_m = df_panel_gr[[c for c in needed if c in df_panel_gr.columns]].dropna().copy()
    if len(df_m) < 30:
        return None

    region_dummies = pd.get_dummies(df_m['region'], prefix='R', dtype=float)
    ref_col = f'R_{_REF_REGION_A}'
    if ref_col in region_dummies.columns:
        region_dummies = region_dummies.drop(columns=[ref_col])

    interact_parts = []
    interact_names = []
    for lag in range(max_lag + 1):
        lc = f'{t_gr_col}_L{lag}'
        for rc in region_dummies.columns:
            interact_parts.append((df_m[lc].values * region_dummies[rc].values).reshape(-1, 1))
            interact_names.append(f'{lc}_x_{rc}')

    country_fe = pd.get_dummies(df_m['country'], prefix='C', drop_first=True, dtype=float)

    X_parts = [df_m[lag_cols].values]
    col_names = lag_cols.copy()
    if interact_parts:
        X_parts.append(np.hstack(interact_parts))
        col_names += interact_names
    X_parts += [country_fe.values, np.ones((len(df_m), 1))]
    col_names += list(country_fe.columns) + ['const']

    X = np.hstack(X_parts)
    y = df_m[e_gr_col].values
    col_idx = {name: i for i, name in enumerate(col_names)}

    coef, se, pval = _ols_hc1(X, y)
    if coef is None:
        return None

    results = {}
    for reg in _REGIONS_A:
        results[reg] = {}
        for lag in range(max_lag + 1):
            base_col = f'{t_gr_col}_L{lag}'
            bi = col_idx.get(base_col)
            beta_k  = coef[bi]  if bi is not None else np.nan
            se_beta = se[bi]    if bi is not None else np.nan
            p_beta  = pval[bi]  if bi is not None else np.nan

            if reg == _REF_REGION_A:
                total_coef, total_se, total_p = beta_k, se_beta, p_beta
            else:
                rc = f'R_{reg}'
                ic = f'{base_col}_x_{rc}'
                ii = col_idx.get(ic)
                if ii is not None:
                    delta    = coef[ii]; se_d = se[ii]; p_d = pval[ii]
                    total_coef = beta_k + delta if not (np.isnan(beta_k) or np.isnan(delta)) else np.nan
                    total_se   = np.sqrt(se_beta**2 + se_d**2) if not (np.isnan(se_beta) or np.isnan(se_d)) else np.nan
                    total_p    = p_d
                else:
                    total_coef, total_se, total_p = np.nan, np.nan, np.nan

            results[reg][lag] = {'coef': total_coef, 'se': total_se, 'p': total_p}
    return results


# Merge records using the shared country-year keys.
_df_nat_gr_all = _df_nat_gr_m.merge(
    _df_nat_gr_i,
    on=['country', 'year'] + [f'{tc}_gr' for tc in _t_cols if f'{tc}_gr' in _df_nat_gr_m.columns and f'{tc}_gr' in _df_nat_gr_i.columns],
    how='outer'
)

_df_nat_gr_all = _df_nat_gr_all.merge(
    df_panel_macro[['country', 'region']].drop_duplicates(),
    on='country', how='left'
)

_trans_gr_cols_A = [f'{c}_gr' for c in _t_cols]
_econ_gr_cols_A = [f'{c}_gr' for c in _e_cols]
_df_A_gr = _build_lag_panel_A_gr(_df_nat_gr_all, _trans_gr_cols_A, _econ_gr_cols_A)

_A_results_gr = {}
for e_col in _e_cols:
    e_gr_col = f'{e_col}_gr'
    for t_col in _t_cols:
        t_gr_col = f'{t_col}_gr'
        _A_results_gr[(e_col, t_col)] = _run_region_interact_model_gr(_df_A_gr, t_gr_col, e_gr_col)



print("Model A estimation completed (growth rates; no year fixed effects) ✓")
print(f"Valid results: {sum(v is not None for v in _A_results_gr.values())}/{len(_A_results_gr)}")
_cmap_A = plt.cm.RdBu_r
_all_coefs_A = [res[reg][lag]['coef']
                for (e_col, t_col), res in _A_results_gr.items() if res is not None
                for reg in _REGIONS_A for lag in range(MAX_LAG_A + 1)
                if not np.isnan(res[reg][lag]['coef'])]
_vmax_A = np.percentile(np.abs(_all_coefs_A), 95) if _all_coefs_A else 1.0
_norm_A = Normalize(vmin=-_vmax_A, vmax=_vmax_A)

fig_A, axes_A = plt.subplots(
    _n_e, _n_t,
    figsize=(2.2 * _n_t + 1.8, 1.5 * _n_e + 0.8),
    gridspec_kw={'hspace': 0.55, 'wspace': 0.30}
)
fig_A.patch.set_facecolor('white')

for ri, e_col in enumerate(_e_cols):
    for ci, t_col in enumerate(_t_cols):
        ax = axes_A[ri, ci]
        res = _A_results_gr.get((e_col, t_col))

        coef_mat = np.full((len(_REGIONS_A), MAX_LAG_A + 1), np.nan)
        pval_mat = np.full((len(_REGIONS_A), MAX_LAG_A + 1), np.nan)
        if res is not None:
            for rri, reg in enumerate(_REGIONS_A):
                for lag in range(MAX_LAG_A + 1):
                    coef_mat[rri, lag] = res[reg][lag]['coef']
                    pval_mat[rri, lag] = res[reg][lag]['p']

        ax.imshow(coef_mat, aspect='auto', cmap=_cmap_A, norm=_norm_A, interpolation='nearest')

        for rri in range(len(_REGIONS_A)):
            for lag in range(MAX_LAG_A + 1):
                pv = pval_mat[rri, lag]; cv = coef_mat[rri, lag]
                if not np.isnan(pv) and pv < 0.05:
                    bg = _cmap_A(_norm_A(cv if not np.isnan(cv) else 0))
                    fc = 'white' if (0.299*bg[0]+0.587*bg[1]+0.114*bg[2]) < 0.45 else '#111'
                    ax.text(lag, rri, '*', ha='center', va='center',
                            fontsize=12, color=fc, fontweight='bold')

        ax.set_xticks(range(MAX_LAG_A + 1))
        ax.set_yticks(range(len(_REGIONS_A)))
        ax.set_xticklabels(_LAG_LABELS, fontsize=9)
        if ci == 0:
            ax.set_yticklabels([_REGION_ABBR[r] for r in _REGIONS_A], fontsize=9)
            ax.set_ylabel(_ECON_LABEL_A.get(e_col, e_col), fontsize=11, fontweight='bold')
        else:
            ax.set_yticklabels([])
        if ri == 0:
            ax.set_title(_TRANS_LABEL_A.get(t_col, t_col), fontsize=11, fontweight='bold', pad=5)
        for sp in ax.spines.values():
            sp.set_linewidth(0.4)

_sm_A = plt.cm.ScalarMappable(cmap=_cmap_A, norm=_norm_A)
_sm_A.set_array([])
_cbar_ax_A = fig_A.add_axes([0.93, 0.15, 0.016, 0.70])
_cb_A = fig_A.colorbar(_sm_A, cax=_cbar_ax_A)
_cb_A.set_label('Total Effect Coef. (Growth Rate)', fontsize=10, labelpad=6, fontweight='bold')
_cb_A.ax.tick_params(labelsize=9)

fig_A.suptitle(
    'Method A (Growth Rate)  ·  Region-Interaction Country FE Panel Model\n'
    r'$\Delta y_{ct}=\sum_k\beta_k \Delta X_{c,t-k}+\sum_r\sum_k\delta_{rk}(\Delta X_{c,t-k}\times Region_r)+\mu_c+\varepsilon_{ct}$',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout(rect=[0, 0, 0.92, 1])
plt.show()


In [ ]:
# Cell | Model B: nested regional heterogeneity

from matplotlib.lines import Line2D as _Line2D

# Estimate region-specific lagged effects for the nested-heterogeneity specification.
MAX_LAG_B = 3

_lag_xs = [0, 1, 2, 3]
_SCALE_COLORS = {
    'continental': '#E63946',
    'regional':    '#457B9D',
    'national':    '#2A9D8F'
}
_SCALE_LS = {
    'continental': '-',
    'regional':    '--',
    'national':    '-.'
}
_SCALE_LABELS = {
    'continental': 'Continental',
    'regional':    'Regional',
    'national':    'National'
}

def _build_multiscale_panel_gr(t_cols, e_col, max_lag=MAX_LAG_B):
    """Helper function."""
    if e_col in _macro_cols:
        df_nat_gr = _df_nat_gr_m.copy()
        df_reg_gr = _df_reg_gr_m.copy()
        df_cont_gr = _df_africa_gr_m.copy()
    else:
        df_nat_gr = _df_nat_gr_i.copy()
        df_reg_gr = _df_reg_gr_i.copy()
        df_cont_gr = _df_africa_gr_i.copy()

    e_gr_col = f'{e_col}_gr'

    if 'region' not in df_nat_gr.columns:
        df_nat_gr = df_nat_gr.merge(
            df_panel_macro[['country', 'region']].drop_duplicates(),
            on='country', how='left'
        )

    # Estimate the model for each transport indicator.
    for tc in t_cols:
        tc_gr = f'{tc}_gr'
        if tc_gr not in df_nat_gr.columns:
            continue
        for lag in range(max_lag + 1):
            df_nat_gr[f'{tc_gr}_nat_L{lag}'] = df_nat_gr.groupby('country')[tc_gr].shift(lag)

    if e_gr_col in df_nat_gr.columns:
        df_nat_gr[f'{e_gr_col}_lag1'] = df_nat_gr.groupby('country')[e_gr_col].shift(1)

    # Estimate the model for each transport indicator.
    for tc in t_cols:
        tc_gr = f'{tc}_gr'
        if tc_gr not in df_reg_gr.columns:
            continue
        for lag in range(max_lag + 1):
            df_reg_gr[f'{tc_gr}_reg_L{lag}'] = df_reg_gr.groupby('region')[tc_gr].shift(lag)

    # Estimate the model for each transport indicator.
    for tc in t_cols:
        tc_gr = f'{tc}_gr'
        if tc_gr not in df_cont_gr.columns:
            continue
        for lag in range(max_lag + 1):
            df_cont_gr[f'{tc_gr}_cont_L{lag}'] = df_cont_gr[tc_gr].shift(lag)

    cont_lag_cols = [c for c in df_cont_gr.columns if '_cont_L' in c]
    df_cont_merge = df_cont_gr[['year'] + cont_lag_cols].copy()

    reg_lag_cols = [c for c in df_reg_gr.columns if '_reg_L' in c]
    df_reg_merge = df_reg_gr[['region', 'year'] + reg_lag_cols].copy()

    df_ms = df_nat_gr.merge(df_cont_merge, on='year', how='left')
    df_ms = df_ms.merge(df_reg_merge, on=['region', 'year'], how='left')

    return df_ms


def _run_multiscale_model_gr(df_ms_gr, t_col, e_col, max_lag=MAX_LAG_B):
    """Helper function."""
    t_gr = f'{t_col}_gr'
    e_gr = f'{e_col}_gr'

    cont_cols = [f'{t_gr}_cont_L{k}' for k in range(max_lag + 1)]
    reg_cols  = [f'{t_gr}_reg_L{k}'  for k in range(max_lag + 1)]
    nat_cols  = [f'{t_gr}_nat_L{k}'  for k in range(max_lag + 1)]
    dyn_col   = f'{e_gr}_lag1'

    avail = [c for c in [e_gr, 'country', 'year', dyn_col] + cont_cols + reg_cols + nat_cols
             if c in df_ms_gr.columns]

    df_m = df_ms_gr[avail].dropna().copy()
    if len(df_m) < 40:
        return None

    country_fe = pd.get_dummies(df_m['country'], prefix='C', drop_first=True, dtype=float)

    indep = ([dyn_col] if dyn_col in df_m.columns else []) + \
            [c for c in cont_cols + reg_cols + nat_cols if c in df_m.columns]

    X = np.hstack([df_m[indep].values, country_fe.values,
                   np.ones((len(df_m), 1))])

    col_names = indep + list(country_fe.columns) + ['const']
    col_idx   = {name: i for i, name in enumerate(col_names)}

    coef, se, pval = _ols_hc1(X, df_m[e_gr].values)
    if coef is None:
        return None

    out = {'continental': {}, 'regional': {}, 'national': {}}
    for scale, cols in [('continental', cont_cols), ('regional', reg_cols), ('national', nat_cols)]:
        for lag, col in enumerate(cols):
            i = col_idx.get(col)
            if i is not None:
                out[scale][lag] = {'coef': coef[i], 'se': se[i], 'p': pval[i]}
            else:
                out[scale][lag] = {'coef': np.nan, 'se': np.nan, 'p': np.nan}

    y_pred = X @ coef
    y_true = df_m[e_gr].values
    n = len(y_true)
    k = X.shape[1]
    resid = y_true - y_pred
    ss_res = np.sum(resid ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k) if (n - k) > 0 else 0
    out['r2'] = max(0, adj_r2)
    out['n_obs'] = n
    return out


_B_results_gr = {}
for e_col in _e_cols:
    df_ms_B_gr = _build_multiscale_panel_gr(_t_cols, e_col)
    for t_col in _t_cols:
        _B_results_gr[(e_col, t_col)] = _run_multiscale_model_gr(df_ms_B_gr, t_col, e_col)


# Initialize the figure layout.
print("Model B estimation completed (growth rates; no year fixed effects) ✓")
print(f"Valid results: {sum(v is not None for v in _B_results_gr.values())}/{len(_B_results_gr)}")
fig_B, axes_B = plt.subplots(
    _n_e, _n_t,
    figsize=(2.4 * _n_t + 2.0, 2.0 * _n_e + 1.0),
    gridspec_kw={'hspace': 0.65, 'wspace': 0.35}
)
fig_B.patch.set_facecolor('white')

for ri, e_col in enumerate(_e_cols):
    for ci, t_col in enumerate(_t_cols):
        ax = axes_B[ri, ci]
        res = _B_results_gr.get((e_col, t_col))

        ax.axhline(0, color='#aaaaaa', linewidth=0.8, linestyle='-', zorder=0)
        ax.set_facecolor('#fafafa')

        if res is not None:
            for scale in ['continental', 'regional', 'national']:
                coefs = [res[scale][lag]['coef'] for lag in _lag_xs]
                pvals = [res[scale][lag]['p']    for lag in _lag_xs]
                valid = [(x, c, p) for x, c, p in zip(_lag_xs, coefs, pvals) if not np.isnan(c)]
                if len(valid) < 2: continue
                xs_v, ys_v, ps_v = zip(*valid)

                ax.plot(xs_v, ys_v,
                        color=_SCALE_COLORS[scale], linestyle=_SCALE_LS[scale],
                        linewidth=1.8, marker='o', markersize=5, markeredgewidth=0,
                        label=_SCALE_LABELS[scale], zorder=3)

                span = max(ys_v) - min(ys_v) if len(ys_v) > 1 else 1
                for x, y, p in zip(xs_v, ys_v, ps_v):
                    if p < 0.05:
                        ax.text(x, y + span * 0.18, '*', ha='center', va='bottom',
                                fontsize=11, color=_SCALE_COLORS[scale], fontweight='bold')

        ax.set_xlim(-0.3, 3.3)
        ax.set_xticks(_lag_xs)
        ax.set_xticklabels(['L0', 'L1', 'L2', 'L3'], fontsize=9)
        ax.tick_params(axis='y', labelsize=8)
        for sp in ax.spines.values():
            sp.set_linewidth(0.4); sp.set_color('#cccccc')

        if ci == 0:
            ax.set_ylabel(_ECON_LABEL_A.get(e_col, e_col), fontsize=11, fontweight='bold')
        else:
            ax.set_yticklabels([])
        if ri == 0:
            ax.set_title(_TRANS_LABEL_A.get(t_col, t_col), fontsize=11, fontweight='bold', pad=4)
        if ri < _n_e - 1:
            ax.set_xticklabels([])

_legend_h = [_Line2D([0],[0], color=_SCALE_COLORS[s], linestyle=_SCALE_LS[s],
                      linewidth=2, marker='o', markersize=5, label=_SCALE_LABELS[s])
             for s in ['continental', 'regional', 'national']]

fig_B.legend(handles=_legend_h, loc='center right', bbox_to_anchor=(0.99, 0.5),
             fontsize=10, frameon=True, framealpha=0.9, edgecolor='#cccccc',
             title='Scale', title_fontsize=10)

fig_B.suptitle(
    'Method B (Growth Rate)  ·  Multi-Scale Hierarchical Nested Model (Country FE only)\n'
    r'$\Delta y_{ct}=\rho \Delta y_{c,t-1}+\sum_k\beta_k \Delta X^{cont}_{t-k}+\sum_k\theta_k \Delta X^{reg}_{r,t-k}+\sum_k\phi_k \Delta X^{nat}_{c,t-k}+v_c+\varepsilon_{ct}$',
    fontsize=12, fontweight='bold', y=1.02
)

plt.tight_layout(rect=[0, 0, 0.87, 1])
plt.show()


In [ ]:
# Cell | Nested-model diagnostic assessment
import statsmodels.api as sm
from scipy import stats as sp_stats
from scipy.stats import chi2

print("" + "=" * 80)
print("[Model B: statistical assessment of the nested multilevel model across four dimensions]")
print("=" * 80)

def calculate_icc_from_panel(df_panel, econ_col, entity_col='country', group_col='region'):
    df = df_panel[[econ_col, entity_col, group_col]].dropna()
    if len(df) < 30:
        return None
    grand_mean = df[econ_col].mean()
    group_means = df.groupby(group_col)[econ_col].mean()
    group_sizes = df.groupby(group_col).size()
    between_var = np.sum(group_sizes * (group_means - grand_mean)**2) / (len(group_means) - 1)
    within_var = 0
    total_within_n = 0
    for grp in df[group_col].unique():
        df_grp = df[df[group_col] == grp]
        grp_mean = df_grp[econ_col].mean()
        within_var += np.sum((df_grp[econ_col] - grp_mean)**2)
        total_within_n += len(df_grp) - 1
    within_var = within_var / total_within_n if total_within_n > 0 else 0
    icc = between_var / (between_var + within_var) if (between_var + within_var) > 0 else 0
    return {'icc': icc, 'between_var': between_var, 'within_var': within_var,
            'n_groups': len(group_means), 'n_obs': len(df)}

print("-" * 80)
print("Dimension 1: assessment of the need for multilevel modeling (ICC)")
print("-" * 80)

icc_results = {}
for e_col in _e_cols:
    e_gr_col = f'{e_col}_gr'
    df_use = _df_nat_gr_m.copy() if e_col in _macro_cols else _df_nat_gr_i.copy()
    if 'region' not in df_use.columns:
        df_use = df_use.merge(df_panel_macro[['country', 'region']].drop_duplicates(), on='country', how='left')
    icc_res = calculate_icc_from_panel(df_use, e_gr_col, 'country', 'region')
    if icc_res is not None:
        icc_results[e_col] = icc_res
        print(f"{_ECON_LABEL_A.get(e_col, e_col)}: ICC={icc_res['icc']:.4f}, "
              f"Between-region variance={icc_res['between_var']:.4f}, Within-country variance={icc_res['within_var']:.4f}")
        print(f"  {'✓' if icc_res['icc'] > 0.059 else '✗'} ICC {'>' if icc_res['icc'] > 0.059 else '≤'} 0.059")

avg_icc = np.mean([r['icc'] for r in icc_results.values()])

print(f"[ICC summary] Mean ICC={avg_icc:.4f}, Indicators with ICC>0.059: {sum(1 for r in icc_results.values() if r['icc'] > 0.059)}/{len(icc_results)}")
def calculate_model_fit(X, y, k):
    n = len(y)
    try:
        beta = np.linalg.lstsq(X, y, rcond=None)[0]
        y_pred = X @ beta
        resid = y - y_pred
        sigma2_mle = np.sum(resid**2) / n
        ll = -0.5 * n * (np.log(2 * np.pi) + np.log(sigma2_mle) + 1)
        aic = -2 * ll + 2 * k
        bic = -2 * ll + k * np.log(n)
        return {'ll': ll, '-2ll': -2 * ll, 'aic': aic, 'bic': bic, 'n': n, 'k': k}
    except:
        return None

print("-" * 80)
print("Dimension 2: comparison of overall model fit (-2LL, AIC, BIC; no year fixed effects)")
print("-" * 80)

fit_comparison = {}

for e_col in _e_cols:
    for t_col in _t_cols:
        df_ms_gr = _build_multiscale_panel_gr([t_col], e_col)
        t_gr = f'{t_col}_gr'
        e_gr = f'{e_col}_gr'
        # Estimate region-specific lagged effects for the nested-heterogeneity specification.
        cont_cols = [f'{t_gr}_cont_L{k}' for k in range(MAX_LAG_B + 1)]
        reg_cols  = [f'{t_gr}_reg_L{k}'  for k in range(MAX_LAG_B + 1)]
        nat_cols  = [f'{t_gr}_nat_L{k}'  for k in range(MAX_LAG_B + 1)]
        dyn_col   = f'{e_gr}_lag1'
        avail = [c for c in [e_gr, 'country', 'year', dyn_col] + cont_cols + reg_cols + nat_cols
                 if c in df_ms_gr.columns]
        df_m = df_ms_gr[avail].dropna().copy()
        if len(df_m) < 40:
            continue

        country_fe = pd.get_dummies(df_m['country'], prefix='C', drop_first=True, dtype=float)
        y = df_m[e_gr].values
        dyn_avail = [dyn_col] if dyn_col in df_m.columns else []

        X0 = np.hstack([df_m[dyn_avail].values] if dyn_avail else [np.empty((len(df_m), 0))])
        X0 = np.hstack([X0, country_fe.values, np.ones((len(df_m), 1))])
        fit0 = calculate_model_fit(X0, y, X0.shape[1])

        cont_avail = [c for c in cont_cols if c in df_m.columns]
        fit1 = calculate_model_fit(
            np.hstack([df_m[dyn_avail + cont_avail].values, country_fe.values, np.ones((len(df_m), 1))]),
            y, len(dyn_avail) + len(cont_avail) + country_fe.shape[1] + 1
        ) if cont_avail else None

        reg_avail = [c for c in reg_cols if c in df_m.columns]
        fit2 = calculate_model_fit(
            np.hstack([df_m[dyn_avail + cont_avail + reg_avail].values, country_fe.values, np.ones((len(df_m), 1))]),
            y, len(dyn_avail) + len(cont_avail) + len(reg_avail) + country_fe.shape[1] + 1
        ) if (cont_avail and reg_avail) else None

        nat_avail = [c for c in nat_cols if c in df_m.columns]
        fit3 = calculate_model_fit(
            np.hstack([df_m[dyn_avail + cont_avail + reg_avail + nat_avail].values, country_fe.values, np.ones((len(df_m), 1))]),
            y, len(dyn_avail) + len(cont_avail) + len(reg_avail) + len(nat_avail) + country_fe.shape[1] + 1
        ) if (cont_avail and reg_avail and nat_avail) else None

        if fit0 and fit3:
            fit_comparison[(e_col, t_col)] = {'null': fit0, 'continental': fit1, 'cont_reg': fit2, 'full': fit3}

print(f"Model combinations successfully compared: {len(fit_comparison)}")
for (e_col, t_col), fits in list(fit_comparison.items()):
    print(f"{_ECON_LABEL_A.get(e_col, e_col)} ~ {_TRANS_LABEL_A.get(t_col, t_col)}:")
    for mn in ['null', 'continental', 'cont_reg', 'full']:
        f = fits.get(mn)
        if f:
            print(f"  {mn:<15} n={f['n']:<5} k={f['k']:<4} AIC={f['aic']:.2f} BIC={f['bic']:.2f}{'  ★' if mn=='full' else ''}")
    if fits['null'] and fits['full']:
        print(f"  AIC improvement: {fits['null']['aic']-fits['full']['aic']:.2f}")

all_aic = [fits['null']['aic'] - fits['full']['aic'] for fits in fit_comparison.values() if fits['null'] and fits['full']]


print(f"[Model-fit summary] Mean AIC improvement: {np.mean(all_aic):.2f}, Proportion with positive improvement: {sum(1 for x in all_aic if x>0)/len(all_aic):.1%}")
print("-" * 80)
print("Dimension 3: explanatory contribution by level (variance decomposition based on log-likelihood increments)")
print("-" * 80)

pseudo_r2_results = {}
for (e_col, t_col), fits in fit_comparison.items():
    fit_null, fit_cont, fit_reg, fit_full = [fits.get(k) for k in ['null', 'continental', 'cont_reg', 'full']]
    if not all([fit_null, fit_cont, fit_reg, fit_full]):
        continue
    total_pseudo_r2 = 1 - (fit_full['ll'] / fit_null['ll']) if fit_null['ll'] != 0 else 0
    if total_pseudo_r2 <= 0:
        continue
    d_cont = max(0, fit_cont['ll'] - fit_null['ll'])
    d_reg  = max(0, fit_reg['ll']  - fit_cont['ll'])
    d_nat  = max(0, fit_full['ll'] - fit_reg['ll'])
    total_d = d_cont + d_reg + d_nat
    if total_d > 0:
        pseudo_r2_results[(e_col, t_col)] = {
            'continental_pct': d_cont / total_d * 100,
            'regional_pct':    d_reg  / total_d * 100,
            'national_pct':    d_nat  / total_d * 100,
            'r2': total_pseudo_r2
        }

for e_col in _e_cols:
    res_e = {k: v for k, v in pseudo_r2_results.items() if k[0] == e_col}
    if not res_e: continue


    print(f"{_ECON_LABEL_A.get(e_col, e_col)}: Pseudo-R²={np.mean([v['r2'] for v in res_e.values()]):.4f}")
    print(f"  Continental:{np.mean([v['continental_pct'] for v in res_e.values()]):.1f}%  "
          f"Regional:{np.mean([v['regional_pct'] for v in res_e.values()]):.1f}%  "
          f"National:{np.mean([v['national_pct'] for v in res_e.values()]):.1f}%")
print("-" * 80)
print("Dimension 4: parameter significance and spatial heterogeneity tests")
print("-" * 80)

sig_stats = {s: {'total': 0, 'sig': 0, 'pos_sig': 0, 'neg_sig': 0}
             for s in ['continental', 'regional', 'national']}
heterogeneity_cases = []

for (e_col, t_col), res in _B_results_gr.items():
    if res is None: continue
    scale_effects = {}
    for scale in ['continental', 'regional', 'national']:
        for lag in range(MAX_LAG_B + 1):
            if lag not in res[scale]: continue
            coef = res[scale][lag]['coef']
            pval = res[scale][lag]['p']
            sig_stats[scale]['total'] += 1
            if not np.isnan(pval) and pval < 0.05:
                sig_stats[scale]['sig'] += 1
                if not np.isnan(coef):
                    if coef > 0:
                        sig_stats[scale]['pos_sig'] += 1
                        scale_effects.setdefault(scale, []).append('pos')
                    else:
                        sig_stats[scale]['neg_sig'] += 1
                        scale_effects.setdefault(scale, []).append('neg')
    if len(scale_effects) >= 2:
        has_pos = any('pos' in e for e in scale_effects.values())
        has_neg = any('neg' in e for e in scale_effects.values())
        if has_pos and has_neg:
            heterogeneity_cases.append({'econ': e_col, 'transport': t_col, 'effects': scale_effects})

print(f"{'Scale':<12} {'Total':<8} {'Significant':<8} {'Significance rate':<10} {'Positive':<8} {'Negative':<8}")
print("-" * 60)
for scale in ['continental', 'regional', 'national']:
    s = sig_stats[scale]
    rate = s['sig'] / s['total'] * 100 if s['total'] > 0 else 0
    label = {'continental': 'Continental', 'regional': 'Regional', 'national': 'National'}[scale]
    print(f"{label:<12} {s['total']:<8} {s['sig']:<8} {rate:<10.1f}% {s['pos_sig']:<8} {s['neg_sig']:<8}")

print(f"\nSpatial heterogeneity cases: {len(heterogeneity_cases)} cases")
for case in heterogeneity_cases[:3]:
    print(f"  {_ECON_LABEL_A.get(case['econ'],case['econ'])} ~ {_TRANS_LABEL_A.get(case['transport'],case['transport'])}: "
          + ", ".join(f"{s}={'Positive/negative' if 'pos' in e and 'neg' in e else ('Positive' if 'pos' in e else 'Negative')}"
                      for s, e in case['effects'].items()))

total_coefs = sum(s['total'] for s in sig_stats.values())
total_sig   = sum(s['sig']   for s in sig_stats.values())
print(f"\nOverall significance rate: {total_sig}/{total_coefs} ({total_sig/total_coefs*100:.1f}%)")
print("=" * 80)
print("[Assessment across four dimensions completed]")


In [ ]:
# Cell | Distributional visualization of cumulative effects
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family':        'Arial',
    'font.size':          12,
    'axes.linewidth':     0.8,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'xtick.major.width':  0.8,
    'ytick.major.width':  0.8,
    'xtick.major.size':   3,
    'ytick.major.size':   3,
    'figure.dpi':         150,
    'axes.unicode_minus': False,
})

_ECON_DISPLAY = {
    'GDP':                     'GDP',
    'Agriculture_value_added': 'AVA',
    'Industry_value_added':    'IVA',
    'Services_value_added':    'SVA',
}
_SCALES_LOWER = ['continental', 'regional', 'national']
_SCALES_UPPER = ['Continental', 'Regional', 'National']
x_label = ['Continental', 'Subregional', 'National']
_SCALE_MAP    = dict(zip(_SCALES_LOWER, _SCALES_UPPER))

_rows_cum = []
for e_col in _e_cols:
    econ_label = _ECON_DISPLAY.get(e_col, e_col)
    for t_col in _t_cols:
        res = _B_results_gr.get((e_col, t_col))
        if res is None:
            continue
        for scale_lower in _SCALES_LOWER:
            # Estimate region-specific lagged effects for the nested-heterogeneity specification.
            coefs = [res[scale_lower][lag]['coef'] for lag in range(MAX_LAG_B + 1)]
            valid  = [c for c in coefs if not np.isnan(c)]
            if len(valid) == 0:
                continue
            cum_coef = float(np.sum(valid))
            _rows_cum.append({
                'econ':  econ_label,
                'scale': _SCALE_MAP[scale_lower],
                't_col': t_col,
                'coef':  cum_coef,
            })

df_cum = pd.DataFrame(_rows_cum)
ECON_VARS   = list(_ECON_DISPLAY.values())
SCALE_ORDER = _SCALES_UPPER

TRANSPORT_CFG = {
    'total_road_km':            {'color': '#E64B35', 'marker': 'o', 'label': 'Road',            'size_ratio': 1.0},
    'motorway_trunk_paved_km':  {'color': '#F39B7F', 'marker': 's', 'label': 'Paved Highway',   'size_ratio': 1.0},
    'railway_length_km':        {'color': '#4DBBD5', 'marker': '^', 'label': 'Railway',         'size_ratio': 1.0},
    'railway_station_area_km2': {'color': '#00A087', 'marker': 'D', 'label': 'Railway Station', 'size_ratio': 1.0},
    'port_area_km2':            {'color': '#7E6148', 'marker': 'P', 'label': 'Port',            'size_ratio': 1.2},
    'airport_area_km2':         {'color': '#8491B4', 'marker': '*', 'label': 'Aviation',         'size_ratio': 2.2},
}

SCALE_CFG = {
    'Continental': {'fill': '#FEE0D2', 'edge': '#CB181D'},
    'Regional':    {'fill': '#DEEBF7', 'edge': '#2171B5'},
    'National':    {'fill': '#E5F5E0', 'edge': '#238B45'},
}
SCALE_XPOS = {s: i for i, s in enumerate(SCALE_ORDER)}

def beeswarm_x(y_vals, max_width=0.28, nbins=20):
    y = np.asarray(y_vals, float)
    n = len(y)
    if n <= 1:
        return np.zeros(n)
    lo, hi = y.min() - 1e-9, y.max() + 1e-9
    bins = np.linspace(lo, hi, nbins + 1)
    bidx = np.clip(np.digitize(y, bins) - 1, 0, nbins - 1)
    out  = np.zeros(n)
    for b in range(nbins):
        idx = np.where(bidx == b)[0]
        k   = len(idx)
        if k < 2:
            continue
        spread = np.linspace(-max_width / 2, max_width / 2, k)
        order  = np.argsort(y[idx])
        out[idx[order]] = spread
    return out

p1, p99 = np.percentile(df_cum['coef'].values, [1, 99.9])
pad  = (p99 - p1) * 0.1
ylim = (p1 - pad, p99 + pad)

# ══════════════════════════════════════════════════════════════
# Initialize the figure layout.
# ══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 4, figsize=(14, 2.5), sharey=True,
                          gridspec_kw={'wspace': 0.06})
fig.patch.set_facecolor('white')

for pi, econ in enumerate(ECON_VARS):
    ax   = axes[pi]
    df_e = df_cum[df_cum['econ'] == econ]

    ax.axhline(0, color='#AAAAAA', lw=0.8, ls='--', zorder=0)
    ax.set_facecolor('white')

    for scale in SCALE_ORDER:
        xc   = SCALE_XPOS[scale]
        vals = df_e[df_e['scale'] == scale]['coef'].dropna().values
        cfg  = SCALE_CFG[scale]

        if len(vals) >= 3:
            try:
                kde   = gaussian_kde(vals, bw_method='scott')
                ygrid = np.linspace(vals.min(), vals.max(), 300)
                dens  = kde(ygrid)
                dens  = dens / dens.max() * 0.36
                ax.fill_betweenx(ygrid, xc - dens, xc + dens,
                                  color=cfg['fill'], alpha=0.90, lw=0, zorder=1)
                ax.plot(xc - dens, ygrid, color=cfg['edge'], lw=0.9, alpha=0.8, zorder=2)
                ax.plot(xc + dens, ygrid, color=cfg['edge'], lw=0.9, alpha=0.8, zorder=2)
            except Exception:
                pass

        if len(vals) > 0:
            ax.hlines(np.median(vals), xc - 0.13, xc + 0.13,
                      colors=cfg['edge'], lw=2.2, zorder=5)

    for scale in SCALE_ORDER:
        xc   = SCALE_XPOS[scale]
        df_s = df_e[df_e['scale'] == scale].reset_index(drop=True)
        if df_s.empty:
            continue
        xoff = beeswarm_x(df_s['coef'].values, max_width=0.26, nbins=15)
        for i, row in df_s.iterrows():
            cfg = TRANSPORT_CFG.get(row['t_col'], {'color': '#888888', 'marker': 'o', 'size_ratio': 1.0})
            
            point_size = 48 * cfg.get('size_ratio', 1.0)
            
            ax.scatter(xc + xoff[i], row['coef'],
                       c=cfg['color'], marker=cfg['marker'],
                       s=point_size, linewidths=0.5, edgecolors='white',
                       zorder=6, alpha=0.92)

    ax.set_xlim(-0.65, 2.65)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(x_label, fontsize=10)
    ax.set_ylim(ylim)
    ax.tick_params(axis='y', labelsize=8)
    ax.set_title(econ, fontsize=14, fontweight='bold', pad=0)
    if pi == 0:
        ax.set_ylabel('Cumulative Effect',
                       fontsize=11, labelpad=4)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)
    for sp in ['left', 'bottom']:
        ax.spines[sp].set_linewidth(0.8)
    
    ax.text(0.02, 1.09, chr(ord('H')+pi), weight='bold',
            transform=ax.transAxes, fontsize=16,
            va='top', ha='left', color='#333333')

legend_handles = [
    Line2D([0], [0], marker=cfg['marker'], color='w',
           markerfacecolor=cfg['color'], markeredgecolor='white',
           markersize=7.5 * (cfg.get('size_ratio', 1.0) ** 0.5), 
           label=cfg['label'], linestyle='None')
    for tc, cfg in TRANSPORT_CFG.items() if tc in _t_cols
]

fig.legend(
    handles=legend_handles,
    fontsize=12, loc='right',
    bbox_to_anchor=(1.02, 0.5), ncol=1,
    frameon=True, framealpha=0.95, edgecolor='#CCCCCC',
    handletextpad=0.5, columnspacing=1.2,
)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()
